# 卷积神经网络
1. ```nn.Sequential```：类似于机器学习里的```pipeline```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
import torchvision

EPOCH = 2
BATCH_SIZE = 50

# 加载训练数据
train_data = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=torchvision.transforms.ToTensor(),
    download=True
)
# 载入测试数据
test_data = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=torchvision.transforms.ToTensor(),
    download=True
)

# 创建训练数据加载器
train_loader = Data.DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True # 训练集打乱顺序
)
# 创建测试数据加载器
test_loader = Data.DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False # 测试集不打乱顺序
)

# 设备配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 取部分测试数据进行测试 - 修复：将数据转移到正确的设备上
test_x = torch.unsqueeze(test_data.data, dim=1).type(torch.FloatTensor)[:2000]/255.
test_y = test_data.targets[:2000]
test_x = test_x.to(device)  # 转移到GPU
test_y = test_y.to(device)  # 转移到GPU

# 定义卷积神经网络
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 第一层卷积
        self.conv1 = nn.Sequential(
            # 第一步，特征提取
            nn.Conv2d(
                in_channels=1,      # 输入通道数,即图片的深度,灰度图为1,彩色图为3
                out_channels=32,    # 卷积核数量,即输出通道数
                kernel_size=5,     # 卷积核尺寸
                stride=1,          # 步长
                padding=2,          # 如果想要卷积后尺寸不变,则padding=(kernel_size-1)/2 当stride=1
                dilation=1         # 控制卷积核元素之间的间距
            ),
            # 第二步，激活函数
            nn.ReLU(),
            # 第三步，池化层
            nn.MaxPool2d(kernel_size=2)  # 池化层,卷积后尺寸减半
        )
        # 第二层卷积
        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        # 全连接层
        self.ful1 = nn.Linear(64*7*7, 512)  # 7*7是图片尺寸,64是上层卷积核数量
        self.drop = nn.Dropout(p=0.5)  # Dropout层,防止过拟合
        # 修复：移除Softmax，因为CrossEntropyLoss内部已包含
        self.ful2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)  # 展平多维的卷积图成一维的向量
        x = self.ful1(x)
        x = self.drop(x)
        output = self.ful2(x)
        # 修复：只返回output，不返回中间特征
        return output
    
# 实例化CNN并转移到GPU
cnn = CNN().to(device)
# 初始化优化器
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)    # Adam优化器
# 定义损失函数
loss_func = nn.CrossEntropyLoss()   # 交叉熵损失函数,多分类问题

# 训练模型
for epoch in range(EPOCH):
    for step, (b_x, b_y) in enumerate(train_loader):   # 载入一个批次的数据
        # 将数据转移到GPU
        b_x, b_y = b_x.to(device), b_y.to(device)
        # 计算loss并修正权重
        output = cnn(b_x)  # 修复：现在只返回一个值
        loss = loss_func(output, b_y)  # 计算损失
        optimizer.zero_grad()           # 清空上一步的残余更新参数值
        loss.backward()                 # 误差反向传播,计算参数更新值
        optimizer.step()                # 将参数更新值施加到net的parameters上

        if step % 50 == 0:
            with torch.no_grad():  # 测试时不需要梯度
                test_output = cnn(test_x)                   # 预测值
                # 修复：正确处理GPU tensor和准确率计算
                pred_y = torch.max(test_output, 1)[1]   # 取最大值的索引
                accuracy = (pred_y == test_y).float().mean().item()  # 修复：正确计算准确率
                print('Epoch: ', epoch, '| train loss: %.4f' % loss.item(), '| test accuracy: %.2f' % accuracy)

Epoch:  0 | train loss: 2.3155 | test accuracy: 0.10
Epoch:  0 | train loss: 0.1887 | test accuracy: 0.88
Epoch:  0 | train loss: 0.2081 | test accuracy: 0.91
Epoch:  0 | train loss: 0.0936 | test accuracy: 0.95
Epoch:  0 | train loss: 0.1354 | test accuracy: 0.95
Epoch:  0 | train loss: 0.3909 | test accuracy: 0.96
Epoch:  0 | train loss: 0.0856 | test accuracy: 0.96
Epoch:  0 | train loss: 0.0864 | test accuracy: 0.96
Epoch:  0 | train loss: 0.1340 | test accuracy: 0.96
Epoch:  0 | train loss: 0.0181 | test accuracy: 0.97
Epoch:  0 | train loss: 0.0537 | test accuracy: 0.97
Epoch:  0 | train loss: 0.0370 | test accuracy: 0.96
Epoch:  0 | train loss: 0.0968 | test accuracy: 0.97
Epoch:  0 | train loss: 0.0351 | test accuracy: 0.98
Epoch:  0 | train loss: 0.0500 | test accuracy: 0.97
Epoch:  0 | train loss: 0.0464 | test accuracy: 0.98
Epoch:  0 | train loss: 0.0101 | test accuracy: 0.98
Epoch:  0 | train loss: 0.0222 | test accuracy: 0.96
Epoch:  0 | train loss: 0.1046 | test accuracy

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
import torchvision

EPOCH = 10
BATCH_SIZE = 50

# 加载训练数据 - 修复：使用CIFAR10训练数据
train_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    transform=torchvision.transforms.ToTensor(),
    download=True
)
# 载入测试数据 - 修复：使用CIFAR10测试数据保持一致
test_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    transform=torchvision.transforms.ToTensor(),
    download=True
)

# 创建训练数据加载器
train_loader = Data.DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True # 训练集打乱顺序
)
# 创建测试数据加载器
test_loader = Data.DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False # 测试集不打乱顺序
)

# 设备配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 取部分测试数据进行测试 - 修复：正确处理CIFAR10数据格式
test_x = torch.from_numpy(test_data.data[:2000]).float() / 255.0  # 先转换为tensor再调用float()
test_x = test_x.permute(0, 3, 1, 2)  # 从(N,H,W,C)转换为(N,C,H,W)
test_y = torch.tensor(test_data.targets[:2000])
test_x = test_x.to(device)  # 转移到GPU
test_y = test_y.to(device)  # 转移到GPU

# 定义卷积神经网络
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 第一层卷积
        self.conv1 = nn.Sequential(
            # 第一步，特征提取
            nn.Conv2d(
                in_channels=3,      # 输入通道数,CIFAR10彩色图为3
                out_channels=32,    # 卷积核数量,即输出通道数
                kernel_size=5,     # 卷积核尺寸
                stride=1,          # 步长
                padding=2,          # 如果想要卷积后尺寸不变,则padding=(kernel_size-1)/2 当stride=1
                dilation=1         # 控制卷积核元素之间的间距
            ),
            # 第二步，激活函数
            nn.ReLU(),
            # 第三步，池化层
            nn.MaxPool2d(kernel_size=2)  # 池化层,卷积后尺寸减半 32->16
        )
        # 第二层卷积
        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)  # 16->8
        )
        # 第三层卷积
        self.conv3 = nn.Sequential(
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)  # 8->4
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(
                in_channels=128,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)  # 4->2
        )        
        # 全连接层 - 修复：正确计算输入维度 128*2*2=512
        self.ful1 = nn.Linear(512, 512)  # CIFAR10: 32->16->8->4->2, 128通道, 128*2*2=512
        self.drop = nn.Dropout(p=0.5)  # Dropout层,防止过拟合
        # 修复：移除Softmax，因为CrossEntropyLoss内部已包含
        self.ful2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)  # 修复：使用定义的conv3层
        x = self.conv4(x)  # 使用第四层卷积
        x = x.view(x.size(0), -1)  # 展平多维的卷积图成一维的向量
        x = self.ful1(x)
        x = self.drop(x)
        output = self.ful2(x)
        # 修复：只返回output，不返回中间特征
        return output
    
# 实例化CNN并转移到GPU
cnn = CNN().to(device)
# 初始化优化器
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)    # Adam优化器
# 定义损失函数
loss_func = nn.CrossEntropyLoss()   # 交叉熵损失函数,多分类问题

# 训练模型
for epoch in range(EPOCH):
    for step, (b_x, b_y) in enumerate(train_loader):   # 载入一个批次的数据
        # 将数据转移到GPU
        b_x, b_y = b_x.to(device), b_y.to(device)
        # 计算loss并修正权重
        output = cnn(b_x)  # 修复：现在只返回一个值
        loss = loss_func(output, b_y)  # 计算损失
        optimizer.zero_grad()           # 清空上一步的残余更新参数值
        loss.backward()                 # 误差反向传播,计算参数更新值
        optimizer.step()                # 将参数更新值施加到net的parameters上

        if step % 50 == 0:
            with torch.no_grad():  # 测试时不需要梯度
                test_output = cnn(test_x)                   # 预测值
                # 修复：正确处理GPU tensor和准确率计算
                pred_y = torch.max(test_output, 1)[1]   # 取最大值的索引
                accuracy = (pred_y == test_y).float().mean().item()  # 修复：正确计算准确率
                print('轮次: ', epoch, '| 训练集损失: %.4f' % loss.item(), '| 测试集正确率: %.2f' % accuracy)

轮次:  0 | 训练集损失: 2.3044 | 测试集正确率: 0.11
轮次:  0 | 训练集损失: 2.0312 | 测试集正确率: 0.23
轮次:  0 | 训练集损失: 1.9175 | 测试集正确率: 0.33
轮次:  0 | 训练集损失: 1.6465 | 测试集正确率: 0.35
轮次:  0 | 训练集损失: 1.6675 | 测试集正确率: 0.37
轮次:  0 | 训练集损失: 1.5145 | 测试集正确率: 0.38
轮次:  0 | 训练集损失: 1.4012 | 测试集正确率: 0.42
轮次:  0 | 训练集损失: 1.5929 | 测试集正确率: 0.42
轮次:  0 | 训练集损失: 1.5240 | 测试集正确率: 0.44
轮次:  0 | 训练集损失: 1.5908 | 测试集正确率: 0.43
轮次:  0 | 训练集损失: 1.3642 | 测试集正确率: 0.45
轮次:  0 | 训练集损失: 1.2448 | 测试集正确率: 0.47
轮次:  0 | 训练集损失: 1.3807 | 测试集正确率: 0.50
轮次:  0 | 训练集损失: 1.2818 | 测试集正确率: 0.51
轮次:  0 | 训练集损失: 1.5692 | 测试集正确率: 0.51
轮次:  0 | 训练集损失: 1.4332 | 测试集正确率: 0.53
轮次:  0 | 训练集损失: 1.3056 | 测试集正确率: 0.52
轮次:  0 | 训练集损失: 1.3633 | 测试集正确率: 0.55
轮次:  0 | 训练集损失: 1.3885 | 测试集正确率: 0.55
轮次:  0 | 训练集损失: 1.1836 | 测试集正确率: 0.55
轮次:  1 | 训练集损失: 1.0851 | 测试集正确率: 0.57
轮次:  1 | 训练集损失: 1.4484 | 测试集正确率: 0.55
轮次:  1 | 训练集损失: 1.2463 | 测试集正确率: 0.57
轮次:  1 | 训练集损失: 1.4049 | 测试集正确率: 0.55
轮次:  1 | 训练集损失: 1.2180 | 测试集正确率: 0.58
轮次:  1 | 训练集损失: 0.9278 | 测试集正确率: 0.57
轮次:  1 | 训练集

## 优化版本的卷积神经网络
使用以下优化技术提高模型性能：
1. **数据增强**：随机水平翻转、随机裁剪、颜色抖动
2. **批标准化**：加速训练收敛，提高稳定性
3. **残差连接**：缓解梯度消失问题
4. **学习率调度**：动态调整学习率
5. **更深的网络结构**：增加网络容量
6. **更好的正则化**：使用不同的Dropout策略

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
import torchvision
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import StepLR
import numpy as np

# 超参数配置
EPOCH_OPTIMIZED = 10  # 增加训练轮数
BATCH_SIZE_OPTIMIZED = 128  # 增加批次大小
LEARNING_RATE = 0.001

# 数据增强和预处理
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),  # 随机水平翻转
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # 颜色抖动
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))  # CIFAR-10标准化
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# 重新加载数据集（使用数据增强）
train_data_optimized = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    transform=transform_train,
    download=True
)

test_data_optimized = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    transform=transform_test,
    download=True
)

# 创建数据加载器
train_loader_optimized = Data.DataLoader(
    dataset=train_data_optimized,
    batch_size=BATCH_SIZE_OPTIMIZED,
    shuffle=True,
    num_workers=2  # 多进程加载数据
)

test_loader_optimized = Data.DataLoader(
    dataset=test_data_optimized,
    batch_size=BATCH_SIZE_OPTIMIZED,
    shuffle=False,
    num_workers=2
)

# 残差块定义
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 如果输入输出通道数不同，需要调整残差连接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual  # 残差连接
        out = F.relu(out)
        return out

# 优化版CNN定义
class OptimizedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(OptimizedCNN, self).__init__()
        
        # 初始卷积层
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # 残差层组
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        
        # 全局平均池化
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 分类器
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, num_classes)
        
        # 权重初始化
        self._initialize_weights()
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# 实例化优化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimized_cnn = OptimizedCNN().to(device)

# 打印模型参数数量
total_params = sum(p.numel() for p in optimized_cnn.parameters())
print(f"优化模型总参数量: {total_params:,}")

# 优化器和学习率调度器
optimizer_optimized = torch.optim.Adam(optimized_cnn.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = StepLR(optimizer_optimized, step_size=30, gamma=0.1)  # 每30个epoch学习率减少10倍

# 损失函数
criterion = nn.CrossEntropyLoss()

# 训练函数
def train_optimized_model():
    best_acc = 0
    train_losses = []
    train_accuracies = []
    
    for epoch in range(EPOCH_OPTIMIZED):
        optimized_cnn.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (inputs, targets) in enumerate(train_loader_optimized):
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer_optimized.zero_grad()
            outputs = optimized_cnn(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer_optimized.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += predicted.eq(targets.data).cpu().sum().item()
            
            if batch_idx % 100 == 0:
                print(f'Epoch: {epoch+1}/{EPOCH_OPTIMIZED} | Batch: {batch_idx+1} | '
                      f'Loss: {loss.item():.4f} | Acc: {100.*correct/total:.2f}%')
        
        # 每个epoch结束后评估
        train_acc = 100. * correct / total
        train_loss = running_loss / len(train_loader_optimized)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        
        # 测试评估
        test_acc = evaluate_optimized_model()
        
        # 更新学习率
        scheduler.step()
        
        print(f'Epoch: {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%')
        print(f'当前学习率: {optimizer_optimized.param_groups[0]["lr"]:.6f}')
        print('-' * 80)
        
        # 保存最佳模型
        if test_acc > best_acc:
            best_acc = test_acc
            print(f'新的最佳准确率: {best_acc:.2f}%')
    
    return train_losses, train_accuracies, best_acc

# 评估函数
def evaluate_optimized_model():
    optimized_cnn.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader_optimized:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = optimized_cnn(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += predicted.eq(targets.data).cpu().sum().item()
    
    return 100. * correct / total

# 开始训练
print("开始训练优化版本的CNN...")
print(f"设备: {device}")
print(f"数据增强: 随机裁剪、水平翻转、颜色抖动")
print(f"网络特性: 残差连接、批标准化、全局平均池化")
print("=" * 80)

train_losses, train_accuracies, best_acc = train_optimized_model()

print("=" * 80)
print(f"训练完成！最佳测试准确率: {best_acc:.2f}%")

优化模型总参数量: 11,173,962
开始训练优化版本的CNN...
设备: cuda
数据增强: 随机裁剪、水平翻转、颜色抖动
网络特性: 残差连接、批标准化、全局平均池化
Epoch: 1/10 | Batch: 1 | Loss: 2.3068 | Acc: 13.28%
Epoch: 1/10 | Batch: 1 | Loss: 2.3068 | Acc: 13.28%
Epoch: 1/10 | Batch: 101 | Loss: 1.5450 | Acc: 31.17%
Epoch: 1/10 | Batch: 101 | Loss: 1.5450 | Acc: 31.17%
Epoch: 1/10 | Batch: 201 | Loss: 1.3495 | Acc: 37.81%
Epoch: 1/10 | Batch: 201 | Loss: 1.3495 | Acc: 37.81%
Epoch: 1/10 | Batch: 301 | Loss: 1.2207 | Acc: 42.59%
Epoch: 1/10 | Batch: 301 | Loss: 1.2207 | Acc: 42.59%
Epoch: 1 | Train Loss: 1.4719 | Train Acc: 45.81% | Test Acc: 59.32%
当前学习率: 0.001000
--------------------------------------------------------------------------------
新的最佳准确率: 59.32%
Epoch: 1 | Train Loss: 1.4719 | Train Acc: 45.81% | Test Acc: 59.32%
当前学习率: 0.001000
--------------------------------------------------------------------------------
新的最佳准确率: 59.32%
Epoch: 2/10 | Batch: 1 | Loss: 1.1157 | Acc: 57.81%
Epoch: 2/10 | Batch: 1 | Loss: 1.1157 | Acc: 57.81%
Epoch: 2/10 | 